First we install einops a library  wich  offers a domain spesific languaje  for tensor operations that are cross frameworks  so they can be understood eather in torch tensorflow or jax.Then xformers is a more eficient atention mechanisms and building blocks and because is more modular, you can  change arquitectures.

In [1]:
#only run thisin a colab env if local use conda in torch env
!pip install einops xformers np 

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 856.1 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 143.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 9.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 752.9 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 13.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 5.1 MB/s eta 0:00:

Now i will import torch library and modules also the lbrary copy and math   

In [36]:
import torch
import numpy as np
import torch.nn as nn
from torch.nn import ModuleList, LayerNorm
import torch.nn.functional as func
import copy
import math
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'device running on {device}')
def _get_clones_(module,n):
    """ the porpuse of this funtion is to duplicate the atention head in to 12 modules with same atchitecture"""
    return ModuleList([copy.deepcopy(module) for i in range (n)])


device running on cuda


 
### Conv1D:
Its a liennear layer  that  act as  the  Y= xW +b but where W has to be transpose because of the convention on how is save to allow easy back propagation implementation, it is used in all other layers  
it has the init and forward functions inside 
### FeedForward layer
this layer consist in 2 linnear and an activation function gelu the numerical parameters picked here are in line on those in the pre train achitecture of gpt2, they can be change but it would mean having to do all training from cero and not just fine tune  the model 


In [37]:
class Conv1(nn.Module):
    '''nx:number of imputs and nf output fetures
        this linnear layer is apply only to the last  dimention of the imput (batch,seqlen,extr,embeding)
        
    '''
    def __init__(self,nx,nf):
        super().__init__()
        self.nf=nf
        # create and initialize wieghts matrix on a gausian distribution with standart deviation of 0.02
        w=torch.empty(nf,nx)# the matrix is created in line with the convention of torch so back prop is easier.
        nn.init.normal(w,std=0.02)
        self.weight=nn.Parameter(w) #makes weights learnable
        self.bias=nn.Parameter(torch.zeros(nf)) # bias vector initialization on zeros
        
    def forward(self,x):
        size_out=x.size()[:-1]+(self.nf,)
        #there is a flattening here and  we create the system of equations  wx+b
        x= torch.addmm(self.bias,x.view(-1,x.size(-1),self.weight))# making the 3 or 4 d  tensor into a matrix 
        x.veiw(*size_out)
        return x
    
        

In [38]:
class FeedForward(nn.Module):
    def __init__(self,dropout,d_inout=768,hidden=768*4):
        """d_inout is the size of the embeding  that will get in and out of this layer
            because the feedforwar is made 2  MLP or linnear layers and a glue activation
            hidden is  the size the embeding will be scalated to in those, factor 4 is a convention from vastwani et al 2017
            
        """
        super().__init__()
        self.l1=Conv1(d_inout,hidden)
        self.l2=Conv1(hidden,d_inout)
        self.activation=func.gelu
        self.dropout = nn.Dropout(dropout)
    def forward(self,x)        :
        return self.dropout(self.l2(self.activation(self.l1(x))))
    

now the code of the Attention class where we subdevide  the atenttion mechanism in to 4 diferent functions 

In [39]:
class Attention (nn.Module):
    def __init__(self,d_model=768,heads=12,d_head=64,n_ctx=1054,bias=True,scale=False):
        """
        d_model: the dimenttions of the token that gets in ,
        heads: number of heads 
        d_heads: dimention of each head
        n_ctx:buffer for bias register
        bias:bool value that decides if use the bias
        scale: the factor of skale sqrt(k) for computer convergency 
        """
        super().__init__()
        self.d_model=d_model
        self.heads=heads
        self.scale=scale
        self.l1=Conv1(d_model,d_model*3)# the 3 here is because QKV are compute in one go 
        self.l2=Conv1(d_model,d_model)# tis will go after an atication layer so l1+l2 dont colaps
        self.activation=nn.Softmax(dim=-1)#that value of dim means to  the last dimention 
        self.dropping=nn.Dropout(p=0.0995)
        #create a lower triangular matrix of ones that is reshape to the espected dimention of the masking in multi head attention
        #the tensor is  not trainable and thats why  we use .register_buffer method
        self.register_buffer('bias',torch.tril(torch.ones(n_ctx,n_ctx)).view(1,1,n_ctx,n_ctx))
        
    def split_heads(self,x):
        """ the proyection of the  dimettion is devided into the heads, and the shape in the new shape 
        of the tensor is  : [batch, n_sequence, nhead,fraction of token]then the axes are permute
        so heach head are compute parallel and independent
        [batch,n_head,nsequence,tokenish]
        """
        #creates a new tuple with  the number  of heads and the piece of the token  for each head
        new_shape=x.size()[:-1]+(self.heads,x.size(-1)//self.heads)
        x= x.view(*new_shape)#the * tells python to umpack the tuple new shape
        return x.permute(0,2,1,3)
    def attention(self,q,k,v,attention_mask=None):
        """
        first the token is proyected into a d_l *3 dimentional space and the proyection is spleted into the heads
        hopping each head will grasp  diferent patterns with in the token
        """
        scores=torch.matmul(q,k.transpose(-2,-1)) # K is transpose so dimenttions agree seclengxheaddim * headdimxseclenq
        if self.scale: 
            scores=scores/math.sqrt(v.size(-1)) #scaling by sqrt(dk)
        nd,ns=scores.size(-2),scores.size(-1) # head,seqlen if i am not wrong
        if attention_mask is not None:
            scores=scores+attention_mask # assuming atention mask values are near to - inf
        scores  = self.activation(scores)# añadiendo los valores de softmax
        scores  = self.dropout(scores) # función de dropout 0.1
        outputs = torch.matmul(scores, v) # Multiplicación final del puntaje por V.
         
        return outputs
    def merge_heads(self,x):
        """ it takes all heads after the atenttion layer and reform the token"""
        x=x.permute(0,2,1,3).contiguos()
        #this last method ensures the tensor keeps having each subdivition one next to the other in memory
        new_shape=x.size()[:-2]+(x.size(-2)*x.size(-1),) #removes  the n head dimention and re builds  the token
        return x.view(*new_shape)
    def forward (self,x):
        """here x Has to be the embeded paragraph, first dimention called batch refers in to the sentence in the 
        paragraph, second dimention called sequenze length is how many tokens are in the phraze and at the end the tokens are  
        proyected to a space 64 dimentions they will be computed with q k and v at the same time 
        """
        #linear layer splits  a vector x at the en althoug it has a matrix  W inside 
        x   =self.l1(x) #input gets remap to  - `[1,3,2304] by the firs linnear layes
        q,k,v= x.split(self.d_model,dim=2)# divide el tensor en 3 tensors de tramaño dmodel
        #now the enriched representation of the tocken will be reshaped and the model will learn to work
        # whith  this reshaping
        q,k,v=self.split_heads(q),self.split_heads(k),self.split_heads(v) 
        out= self.attention(q,k,v)
        out=self.merge_heads(out)
        out=self.l2(out) #  it is map to the originall dimention again
        return out

Now we create the trandformer block were we put together attention  feedforward and norm layers.
then we create the GPT class where we put together the transformer intansiation and use the get clones  function to clone that block use embedings cross entropy and layer normalizations
### LayerNorm why ?
Lets remember that normalization is to  map inputs  to the range [0,1] to avoid vanashing or exploiting gradients. Batch norm is to normalize the outputs of every layer . each epoch will take longer but you will need less epocs for the same accuracy.Here the norm is apply to all outputs of each layer inside a batch ( an array of inputs) for layer nor the array of imputs is size 1 . no batches then 

as you can see the gpt is made of transformers the transformers are made of  feedforward and attention layers and both are made of linear layers and activation functions

In [40]:
class Transformer(nn.Module):
    def __init__(self,d_model=768,n_heads=12,dropout=0.0995):
        super().__init__()
        self.attention=Attention(d_model=768,heads=12,d_head=64,n_ctx=1054,bias=True,scale=False)
        self.f_forward=FeedForward(dropout=0.0995,d_inout=768,hidden=768*4)
        self.ln_1=LayerNorm(d_model)
        self.ln_2=LayerNorm(d_model)
    def forward(self,x):
        x=x+self.attention(self.ln_1(x))
        x=x+ self.f_forward(self.ln_2(x))
        return x
    """You add context  from other  tokens and enrich representation but start from the original token never replace it"""

In [48]:
class GPT2(nn.Module):
    def __init__(self,layers=12,n_ctx=1024,d_model=768,vocab_size=50257):
        super(GPT2,self).__init__()
        """
        layers: the quantity  of transformers
        n_ctx: number of tokens from the pass  
        vocab_size= size o the vocab used in the trainning 
        """
        
        self.layers=layers
        block=Transformer(d_model=768,n_heads=12,dropout=0.095)
        self.h=_get_clones_(block,layers)
        self.wte=nn.Embedding(vocab_size, d_model) #word token embeding layer
        self.wpe=nn.Embedding(n_ctx, d_model) #positional embedings
        self.drop=nn.Dropout(0.095)
        self.ln=nn.LayerNorm(d_model)# layer norm  befor logit funtion
        self.out=nn.Linear(d_model,vocab_size,bias=False) #logits for each token in the vocab space
        self.loss=nn.CrossEntropyLoss()
        self.init_weight()
    def init_weight(self):
        """it defines inicialization  rules for weights in one ore many layer types"""
        self.out.weight=self.wte.weight #re using a matrix of weigths 
        self.apply(self._init_weights_) #applys the funtion to all sublayers in the architecture
    def _init_weights_(self,module):
        if isinstance (module,(nn.Linear,nn.Embedding, Conv1)):
            module.weight.data.normal_(mean=0,std=0.02)
        if isinstance(module, (nn.Linear, Conv1)) and module.bias is not None:
            module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

def forward(self, src, labels=None, pos_ids=None):        
    """Forward pass: token + positional embeddings → transformer blocks → logits → optional loss"""

    if pos_ids is None:
        pos_ids = torch.arange(0, src.size(-1)).unsqueeze(0).to(src.device)

    # Embedding + dropout
    inp = self.drop(self.wte(src) + self.wpe(pos_ids))

    # Transformer blocks
    for i in range(self.nlayers): 
        inp = self.h[i](inp)

    # Final normalization + projection
    inp    = self.ln_f(inp)
    logits = self.out(inp)
    outputs = (logits, inp)

    # Training mode: compute loss
    if labels is not None:
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = self.loss_fn(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )
        outputs = (loss,) + outputs
        return loss.mean()

    # Inference mode: return logits
    return logits

                

Now i will create the model gpt2 then i will download the weights of an already train gpt2 model  

In [49]:
import torch.nn.functional as F
import time
from transformers import GPT2Tokenizer
model = GPT2()

/tmp/ipython-input-2268699876.py:11: FutureWarning: `nn.init.normal` is now deprecated in favor of `nn.init.normal_`.
  nn.init.normal(w,std=0.02)


In [50]:
!curl --output gpt2-pytorch_model.bin https://s3.amazonaws.com/models.huggingface.co/bert/gpt2-pytorch_model.bin

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  522M  100  522M    0     0  15.1M      0  0:00:34  0:00:34 --:--:-- 16.9M


In [51]:
model_dict = model.state_dict()
state_dict = torch.load("./gpt2-pytorch_model.bin")

old_keys = []
new_keys = []
for key in state_dict.keys():
    if "mlp" in key: #El diccionario de estado para el MLP feedforward debe ser cambiado por mlp
        new_key = key.replace("mlp", "feedforward")
        new_keys.append(new_key)
        old_keys.append(key)
        
        """the why is because  naming convention in the original paper is mlp and here we used feedforward
        if this is not apply the trained weights are not usable"""
for old_key, new_key in zip(old_keys, new_keys):
    state_dict[new_key]=state_dict.pop(old_key)

Now  we calculate  the size of the model and create the generate function to  generate a prediction and  messure the inference time 

In [52]:
total_params = sum(p.numel() for p in model.parameters())
size_bytes = total_params * 4
size_mb = size_bytes / (1024 ** 2)

print(f"El tamaño total de GPT2 sin alteraciones es: {size_bytes} bytes o {size_mb:.2f} MB")
def gen (context, ntok=550):
    start=time.time()
    for _ in range (ntok):#each pass generates a token
        out=model(context)
        logits=out[:,-1,:] #i am teling to take out the middle position that is seq leng 
        indices_to_remove= logits< torch.topk(logits,10)[0][...,-1,None] #all indices that are less than top k  of logits will be remove
        logits=[indices_to_remove]=-np.inf
        next_tok = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1).squeeze(1)#samples one token
        context=torch.cat([context,next_tok.unsqueeze(-1)],dim=-1)#append the tocken  to  context
    end=time.time()
    inference_time=end-start
    return context,inference_time

        

El tamaño total de GPT2 sin alteraciones es: 497759232 bytes o 474.70 MB


In [54]:
out, inference_time = gen(context, ntok=40)
decoded_output = tokenizer.decode(out[0])# the first batch 

NameError: name 'context' is not defined